## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [1]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

c:\Users\Alin\Desktop\ADC\An II\Sem II\AI Engineering\Proiect\echochamber-project-team-5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [5]:
MY_BUBBLE_FILE = "conspirationist.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: conspirationist
Texte: 4


,id,agent,text
0,yt_PI9K4fsNHvg_UgwZ3VabzgKdxQi62W54AaABAg,Conspiraționist,Sufletul Pământului în rezervorul şi motorul b...
1,yt_Z3b91sZQa5I_UgzhcgjqHRJUMUS6rTt4AaABAg,Conspiraționist,Deci apare agresiunea din partea Rusiei în gen...
2,yt_XOejd9J2xqc_Ugws8_zm56ThcnowcAt4AaABAg,Conspiraționist,Votam masiv Nicusor Dan! Ai scris intr-o posta...
3,yt_sXqW2Cnw0DU_UgyFd08cuCXGNjBPclx4AaABAg,Conspiraționist,"Dictatorul Putin nu vrea pace, vrea capitulare..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [6]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Sufletul Pământului în rezervorul şi motorul başinilor dracului.


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [7]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\Alin\Desktop\ADC\An II\Sem II\AI Engineering\Proiect\echochamber-project-team-5\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alin\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|█

Număr texte: 4
Dimensiune embeddings: (4, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [ ]:
# TODO student:
# Bula mea are 4 texte.
# Au fost generați 4 vectori.
# A doua valoare din embeddings.shape reprezintă dimensiunea vectorilor semantici, care este 384 pentru acest model.

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [8]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\conspirationist
Vectori în index: 4


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: da
# index.pkl există: da
# index.ntotal este egal cu numărul de texte: 4

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [9]:
# Text nou introdus în aplicație

input_text = "Alegerile au fost manipulate de interese externe si de sistemul globalist."

In [10]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [12]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.386
Text: Deci apare agresiunea din partea Rusiei în general, dar nu explicit o posibilă agresiune împotriva României. Pe de altă parte, apare „independența solidară”. Nu reiese clar că trebuie să cam uităm de apărarea țării în favoarea unei solidarități militare evident în cadrul Uniunii europene? Nu este un pas în direcția federalizării fix folosind spaima comunitară de Rusia imperială?

Rezultat 2
Scor: 0.266
Text: Votam masiv Nicusor Dan! Ai scris intr-o postare pe facebook ca ai fost la 7 dezbateri la care nu s-a prezentat contracandidatul. Te rog sa ne spui cum gasim si noi acele dezbateri, nu de alta dar dupa ore de cautari am constatat ca nu exista! Efectiv un mincinos. Si asta a fost o strategie de marketing sau cum?

Rezultat 3
Scor: 0.224
Text: Dictatorul Putin nu vrea pace, vrea capitularea necondiționată a ucrainenilor.

Rezultat 4
Scor: 0.09
Text: Sufletul Pământului în rezervorul şi motorul başinilor dracului.

Rezultat 5
Scor: -3.4028234663852886e+38

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

Din cele 5 rezultate, doar 1 rezultat este clar relevant pentru agentul conspirationist. Primul text adica, deoarece vorbeste despre federalizare, frica de Rusia si influenta Uniunii Europene. Rezultatul 2 este partial relevant, pentru ca exprima neincredere si acuza manipulare politica, dar nu este foarte conspirationist.

Textele recuperate exprima doar partial vocea agentului. Primul text se potriveste cel mai bine, dar rezultatele 3 si 5 despre Putin nu exprima vocea agentului conspirationist, iar rezultatul 4 este slab si greu de interpretat.

As elimina rezultatul 4 din bula, deoarece textul este neclar si nu ajuta la definirea agentului. As verifica si duplicatul rezultatului 3/5, pentru ca acelasi text apare de doua ori.